# Spotify Recommendation Analytics

This notebook is the project's main analysis workflow. It loads the five
accepted cleaned datasets, explores Spotify trends, documents or rebuilds the
four popularity-regression experiments, and validates the persisted
content-based recommender.


## 1. Project configuration

All paths are relative to the project root. Normal demonstration mode keeps
the accepted artifacts read-only. Set `REBUILD_MODELS = True` (or the matching
environment variable in a temporary notebook run) to rebuild models under
`generated_outputs/model_artifacts/`. Writing to `models/` also requires the
separate `OVERWRITE_ACCEPTED_MODELS = True` confirmation.


In [ ]:
from pathlib import Path
import os

working_directory = Path.cwd().resolve()
default_root = working_directory if (working_directory / "data").is_dir() else working_directory.parent
PROJECT_ROOT = Path(os.environ.get("SPOTIFY_PROJECT_ROOT", default_root)).resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
GENERATED_OUTPUT_DIR = Path(
    os.environ.get("SPOTIFY_GENERATED_OUTPUT_DIR", PROJECT_ROOT / "generated_outputs")
).resolve()

REBUILD_MODELS = False
if "SPOTIFY_REBUILD_MODELS" in os.environ:
    REBUILD_MODELS = os.environ["SPOTIFY_REBUILD_MODELS"].strip().lower() in {"1", "true", "yes"}

# This must be changed manually as a second confirmation. It is intentionally
# not controlled by an environment variable.
OVERWRITE_ACCEPTED_MODELS = False

TABLE_OUTPUT_DIR = GENERATED_OUTPUT_DIR / "tables"
FIGURE_OUTPUT_DIR = GENERATED_OUTPUT_DIR / "figures"
MODEL_OUTPUT_DIR = (
    MODEL_DIR
    if REBUILD_MODELS and OVERWRITE_ACCEPTED_MODELS
    else GENERATED_OUTPUT_DIR / "model_artifacts"
)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"REBUILD_MODELS={REBUILD_MODELS}")
print(f"Accepted model directory: {MODEL_DIR}")
print(f"Generated output directory: {GENERATED_OUTPUT_DIR}")


## 2. Imports

The notebook uses pandas and NumPy for analysis, Matplotlib for charts, and
scikit-learn for the optional regression and recommendation rebuild.


In [ ]:
import json
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET = "popularity"
RIDGE_ALPHA = 10.0
REGRESSION_AUDIO_FEATURES = [
    "danceability", "energy", "acousticness", "valence", "tempo",
    "loudness", "instrumentalness", "liveness", "speechiness",
]
REGRESSION_EXTENDED_FEATURES = REGRESSION_AUDIO_FEATURES + [
    "duration_ms", "year", "explicit", "key", "mode",
]
RECOMMENDER_FEATURES = [
    "acousticness", "danceability", "energy", "instrumentalness",
    "liveness", "loudness", "speechiness", "tempo", "valence",
]
TREND_FEATURES = ["energy", "danceability", "acousticness", "valence"]

pd.set_option("display.max_columns", 30)


## 3. Load cleaned datasets

The five CSV files are loaded without rewriting them. Any preparation below
happens only in memory.


In [ ]:
DATA_FILES = {
    "tracks": "data_clean.csv",
    "artist_features": "data_by_artist_clean.csv",
    "genre_features": "data_by_genres_clean.csv",
    "year_features": "data_by_year_clean.csv",
    "artist_genres": "data_w_genres_clean.csv",
}

missing_files = [name for name in DATA_FILES.values() if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError("Missing cleaned dataset(s): " + ", ".join(missing_files))

datasets = {
    name: pd.read_csv(DATA_DIR / filename)
    for name, filename in DATA_FILES.items()
}
tracks = datasets["tracks"].copy()
numeric_columns = sorted(set(REGRESSION_EXTENDED_FEATURES + RECOMMENDER_FEATURES + [TARGET]))
for column in numeric_columns:
    tracks[column] = pd.to_numeric(tracks[column], errors="coerce")
tracks["decade"] = (tracks["year"].astype(int) // 10) * 10

print("Loaded:", ", ".join(DATA_FILES.values()))


## 4. Dataset overview

The overview confirms the role and dimensions of each cleaned table.


In [ ]:
dataset_overview = pd.DataFrame(
    [
        {"dataset": name, "rows": len(frame), "columns": frame.shape[1]}
        for name, frame in datasets.items()
    ]
)
dataset_overview.to_csv(TABLE_OUTPUT_DIR / "dataset_overview.csv", index=False)
display(dataset_overview)


## 5. Data validation

Validation checks the essential identity, regression, and recommendation
contracts before analysis continues.


In [ ]:
required_track_columns = [
    "id", "name", "artists", "year", TARGET,
    *REGRESSION_EXTENDED_FEATURES,
    *RECOMMENDER_FEATURES,
]
missing_columns = [column for column in required_track_columns if column not in tracks.columns]
if missing_columns:
    raise ValueError("Track data is missing columns: " + ", ".join(missing_columns))
if any(frame.empty for frame in datasets.values()):
    raise ValueError("One or more cleaned datasets are empty.")
if not tracks["id"].is_unique:
    raise ValueError("Track IDs must be unique.")
if tracks[["id", "name", "artists", *RECOMMENDER_FEATURES]].isna().any().any():
    raise ValueError("Track identity or recommender features contain missing values.")
if not np.isfinite(tracks[RECOMMENDER_FEATURES].to_numpy(dtype=float)).all():
    raise ValueError("Recommender features contain non-finite values.")

missing_summary = (
    tracks.isna().sum().rename("missing_count").to_frame()
    .assign(missing_rate=lambda frame: frame["missing_count"] / len(tracks))
    .sort_values("missing_count", ascending=False)
    .reset_index(names="column")
)
missing_summary.to_csv(TABLE_OUTPUT_DIR / "missing_values.csv", index=False)
print(f"Validation passed for {len(tracks):,} track rows.")
display(missing_summary.head(10))


## 6. Descriptive statistics

These statistics summarize the regression features and popularity target.


In [ ]:
descriptive_statistics = (
    tracks[REGRESSION_EXTENDED_FEATURES + [TARGET]]
    .describe()
    .T
    .reset_index(names="feature")
)
descriptive_statistics.to_csv(TABLE_OUTPUT_DIR / "descriptive_statistics.csv", index=False)
display(descriptive_statistics.round(3))


## 7. Exploratory data analysis

The popularity distribution and leading genre profiles give a compact first
view of the catalog.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(tracks[TARGET].dropna(), bins=25, edgecolor="white")
axes[0].set(title="Distribution of Track Popularity", xlabel="Popularity", ylabel="Tracks")

top_genres = (
    datasets["genre_features"]
    .dropna(subset=["genres_clean"])
    .sort_values(TARGET, ascending=False)
    .head(10)
    .sort_values(TARGET)
)
axes[1].barh(top_genres["genres_clean"], top_genres[TARGET])
axes[1].set(title="Ten Highest-Popularity Genre Profiles", xlabel="Average popularity")
fig.tight_layout()
fig.savefig(FIGURE_OUTPUT_DIR / "exploratory_overview.png", dpi=160, bbox_inches="tight")
plt.show()

top_genres[["genres_clean", TARGET, "energy", "danceability", "valence"]].to_csv(
    TABLE_OUTPUT_DIR / "top_genre_profiles.csv", index=False
)
display(top_genres[["genres_clean", TARGET, "energy", "danceability", "valence"]].sort_values(TARGET, ascending=False))


## 8. Trends by year and decade

Grouping the track table highlights catalog growth and long-run changes in
average audio characteristics.


In [ ]:
decade_summary = (
    tracks.groupby("decade", as_index=False)
    .agg(
        total_tracks=("id", "count"),
        average_popularity=(TARGET, "mean"),
        average_energy=("energy", "mean"),
        average_danceability=("danceability", "mean"),
        average_acousticness=("acousticness", "mean"),
        average_valence=("valence", "mean"),
    )
    .round(4)
)
year_trends = tracks.groupby("year", as_index=False)[TREND_FEATURES].mean()
decade_summary.to_csv(TABLE_OUTPUT_DIR / "decade_summary.csv", index=False)
year_trends.to_csv(TABLE_OUTPUT_DIR / "audio_trends_by_year.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(decade_summary["decade"].astype(str), decade_summary["total_tracks"])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set(title="Tracks by Decade", xlabel="Decade", ylabel="Tracks")
for feature in TREND_FEATURES:
    axes[1].plot(year_trends["year"], year_trends[feature], label=feature)
axes[1].set(title="Average Audio Features by Year", xlabel="Year", ylabel="Average value")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_OUTPUT_DIR / "year_and_decade_trends.png", dpi=160, bbox_inches="tight")
plt.show()
display(decade_summary)


## 9. Correlation analysis

The matrix compares popularity with the extended numeric predictors while
preserving the accepted feature order.


In [ ]:
correlation_features = REGRESSION_EXTENDED_FEATURES + [TARGET]
correlation_matrix = tracks[correlation_features].corr(numeric_only=True)
correlation_matrix.to_csv(TABLE_OUTPUT_DIR / "correlation_matrix.csv")

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlation_matrix, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=70, ha="right")
ax.set_yticks(range(len(correlation_matrix.index)), correlation_matrix.index)
ax.set_title("Correlation Matrix of Spotify Features")
fig.tight_layout()
fig.savefig(FIGURE_OUTPUT_DIR / "correlation_matrix.png", dpi=160, bbox_inches="tight")
plt.show()
display(correlation_matrix[[TARGET]].sort_values(TARGET, ascending=False).round(3))


## 10. Linear Regression experiments

Linear Regression is evaluated with both the nine audio-only features and the
14 extended features. Both use `test_size=0.2` and `random_state=42`.


In [ ]:
ACCEPTED_METRICS = pd.DataFrame(
    [
        {"feature_set": "Audio Only", "model": "Linear Regression", "MAE": 13.1118, "RMSE": 16.3080, "R2": 0.4442},
        {"feature_set": "Audio Only", "model": "Ridge Regression", "MAE": 13.1119, "RMSE": 16.3080, "R2": 0.4442},
        {"feature_set": "Extended", "model": "Linear Regression", "MAE": 7.9820, "RMSE": 10.7308, "R2": 0.7594},
        {"feature_set": "Extended", "model": "Ridge Regression", "MAE": 7.9821, "RMSE": 10.7309, "R2": 0.7594},
    ]
)
display(ACCEPTED_METRICS.loc[ACCEPTED_METRICS["model"] == "Linear Regression"])


## 11. Ridge Regression experiments

The Ridge experiments use `alpha=10.0` with the same split and feature sets.


In [ ]:
print(f"Ridge alpha: {RIDGE_ALPHA}")
display(ACCEPTED_METRICS.loc[ACCEPTED_METRICS["model"] == "Ridge Regression"])


## 12. Model comparison

Normal mode displays the accepted reproducible metrics. Rebuild mode fits all
four experiments in generated output space and compares the new metrics with
the accepted values.


In [ ]:
feature_sets = {
    "Audio Only": REGRESSION_AUDIO_FEATURES,
    "Extended": REGRESSION_EXTENDED_FEATURES,
}
estimators = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE),
}
fitted_regression_models: dict[tuple[str, str], Pipeline] = {}
regression_predictions: dict[tuple[str, str], tuple[pd.Series, np.ndarray]] = {}

if REBUILD_MODELS:
    metric_rows = []
    for feature_set_name, features in feature_sets.items():
        model_frame = tracks.dropna(subset=[TARGET, *features])
        X_train, X_test, y_train, y_test = train_test_split(
            model_frame[features],
            model_frame[TARGET],
            test_size=0.2,
            random_state=RANDOM_STATE,
        )
        for model_name, estimator in estimators.items():
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(estimator)),
            ])
            pipeline.fit(X_train, y_train)
            predictions = pipeline.predict(X_test)
            key = (feature_set_name, model_name)
            fitted_regression_models[key] = pipeline
            regression_predictions[key] = (y_test, predictions)
            metric_rows.append(
                {
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "MAE": mean_absolute_error(y_test, predictions),
                    "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
                    "R2": r2_score(y_test, predictions),
                }
            )
    regression_metrics = pd.DataFrame(metric_rows)
else:
    regression_metrics = ACCEPTED_METRICS.copy()

regression_metrics.to_csv(TABLE_OUTPUT_DIR / "regression_metrics.csv", index=False)
best_row = regression_metrics.loc[regression_metrics["R2"].idxmax()]
best_experiment = f"{best_row['feature_set']} {best_row['model']}"
if best_experiment != "Extended Linear Regression":
    raise AssertionError(f"Unexpected best model: {best_experiment}")

if REBUILD_MODELS:
    comparison = regression_metrics.merge(
        ACCEPTED_METRICS,
        on=["feature_set", "model"],
        suffixes=("_rebuilt", "_accepted"),
    )
    for metric in ["MAE", "RMSE", "R2"]:
        comparison[f"{metric}_difference"] = (
            comparison[f"{metric}_rebuilt"] - comparison[f"{metric}_accepted"]
        )
    display(comparison.round(4))
else:
    display(regression_metrics)
print("Best model: Extended Linear Regression")


## 13. Best-model persistence

Normal mode loads the accepted best model. Rebuild mode saves the newly fitted
winner to the configured generated artifact folder unless the second overwrite
confirmation was deliberately enabled.


In [ ]:
BEST_MODEL_FILENAME = "best_popularity_model.joblib"
if REBUILD_MODELS:
    best_key = (str(best_row["feature_set"]), str(best_row["model"]))
    best_popularity_model = fitted_regression_models[best_key]
    if MODEL_OUTPUT_DIR.resolve() == MODEL_DIR.resolve() and not OVERWRITE_ACCEPTED_MODELS:
        raise PermissionError("Writing accepted models requires OVERWRITE_ACCEPTED_MODELS=True.")
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_popularity_model, MODEL_OUTPUT_DIR / BEST_MODEL_FILENAME)
    print(f"Rebuilt best model saved to: {MODEL_OUTPUT_DIR / BEST_MODEL_FILENAME}")
else:
    best_popularity_model = joblib.load(MODEL_DIR / BEST_MODEL_FILENAME)
    print(f"Loaded accepted best model: {MODEL_DIR / BEST_MODEL_FILENAME}")


## 14. Content-based recommendation system

Each track is represented by nine ordered audio features. A `StandardScaler`
feeds cosine `NearestNeighbors`; catalog row indexes stay aligned with the
fitted matrix. Model index 19611 is used as a deterministic smoke-test seed.


In [ ]:
CATALOG_COLUMNS = [
    "_model_index", "id", "name", "artists", "year", TARGET,
    *RECOMMENDER_FEATURES,
]

def recommend_from_index(
    catalog: pd.DataFrame,
    scaler: StandardScaler,
    neighbors: NearestNeighbors,
    features: list[str],
    model_index: int,
    top_n: int,
) -> pd.DataFrame:
    if model_index < 0 or model_index >= len(catalog):
        raise IndexError("model_index is outside the catalog.")
    if top_n <= 0:
        raise ValueError("top_n must be positive.")
    query = catalog.iloc[model_index]
    transformed = scaler.transform(catalog.iloc[[model_index]][features])
    query_pair = (str(query["name"]), str(query["artists"]))
    pool = min(len(catalog), max(50, top_n * 8))
    while True:
        distances, indexes = neighbors.kneighbors(transformed, n_neighbors=pool)
        seen_pairs = set()
        rows = []
        for distance, index_value in zip(distances[0], indexes[0]):
            index = int(index_value)
            neighbor = catalog.iloc[index]
            pair = (str(neighbor["name"]), str(neighbor["artists"]))
            if index == model_index or str(neighbor["id"]) == str(query["id"]):
                continue
            if pair == query_pair or pair in seen_pairs:
                continue
            seen_pairs.add(pair)
            rows.append(
                {
                    "rank": len(rows) + 1,
                    "recommended_model_index": index,
                    "recommended_name": neighbor["name"],
                    "recommended_artists": neighbor["artists"],
                    "similarity": 1.0 - float(distance),
                }
            )
            if len(rows) == top_n:
                return pd.DataFrame(rows)
        if pool >= len(catalog):
            raise RuntimeError("The catalog cannot provide the requested unique Top N.")
        pool = min(len(catalog), pool * 2)

if REBUILD_MODELS:
    recommender_catalog = tracks.dropna(
        subset=["id", "name", "artists", *RECOMMENDER_FEATURES]
    ).reset_index(drop=True)
    recommender_catalog.insert(0, "_model_index", np.arange(len(recommender_catalog)))
    recommender_catalog = recommender_catalog[CATALOG_COLUMNS].copy()
    recommender_scaler = StandardScaler()
    recommender_matrix = recommender_scaler.fit_transform(
        recommender_catalog[RECOMMENDER_FEATURES]
    )
    recommender_neighbors = NearestNeighbors(metric="cosine")
    recommender_neighbors.fit(recommender_matrix)
    recommender_feature_order = RECOMMENDER_FEATURES.copy()
else:
    recommender_scaler = joblib.load(MODEL_DIR / "recommender_scaler.joblib")
    recommender_neighbors = joblib.load(MODEL_DIR / "nearest_neighbors_recommender.joblib")
    recommender_catalog = pd.read_csv(MODEL_DIR / "recommender_catalog.csv")
    recommender_feature_order = json.loads(
        (MODEL_DIR / "recommender_features.json").read_text(encoding="utf-8")
    )

smoke_recommendations = recommend_from_index(
    recommender_catalog,
    recommender_scaler,
    recommender_neighbors,
    recommender_feature_order,
    model_index=19611,
    top_n=3,
)
display(smoke_recommendations)


## 15. Recommendation validation

These checks enforce feature order, contiguous row alignment, no self-result,
exact Top N, finite similarities, and deterministic repeatability.


In [ ]:
if recommender_feature_order != RECOMMENDER_FEATURES:
    raise AssertionError("The nine recommender features are out of order.")
if recommender_catalog.columns.tolist() != CATALOG_COLUMNS:
    raise AssertionError("The recommender catalog schema is out of order.")
if not np.array_equal(
    recommender_catalog["_model_index"].to_numpy(),
    np.arange(len(recommender_catalog)),
):
    raise AssertionError("Catalog indexes are not contiguous and row-aligned.")
if int(recommender_neighbors.n_samples_fit_) != len(recommender_catalog):
    raise AssertionError("Catalog rows do not match fitted neighbor rows.")
if len(smoke_recommendations) != 3:
    raise AssertionError("The smoke test did not return exact Top 3.")
if 19611 in smoke_recommendations["recommended_model_index"].tolist():
    raise AssertionError("The seed track appeared in its own recommendations.")
if not np.isfinite(smoke_recommendations["similarity"]).all():
    raise AssertionError("Recommendation similarities must be finite.")

repeat_smoke = recommend_from_index(
    recommender_catalog,
    recommender_scaler,
    recommender_neighbors,
    recommender_feature_order,
    model_index=19611,
    top_n=3,
)
if repeat_smoke["recommended_model_index"].tolist() != smoke_recommendations["recommended_model_index"].tolist():
    raise AssertionError("The deterministic seed did not produce repeatable results.")

if not REBUILD_MODELS:
    aligned_matrix = recommender_scaler.transform(
        recommender_catalog[RECOMMENDER_FEATURES]
    )
    if not np.allclose(aligned_matrix, recommender_neighbors._fit_X, rtol=0, atol=1e-12):
        raise AssertionError("The persisted catalog is not row-aligned with the model.")

print(f"Recommendation validation passed for {len(recommender_catalog):,} catalog rows.")


## 16. Saving model artifacts

Accepted artifacts remain untouched in normal mode. Rebuild mode writes the
new scaler, neighbor model, catalog, and feature contract only to the guarded
output location selected in Section 1.


In [ ]:
if REBUILD_MODELS:
    if MODEL_OUTPUT_DIR.resolve() == MODEL_DIR.resolve() and not OVERWRITE_ACCEPTED_MODELS:
        raise PermissionError("Writing accepted models requires the second confirmation.")
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(recommender_scaler, MODEL_OUTPUT_DIR / "recommender_scaler.joblib")
    joblib.dump(recommender_neighbors, MODEL_OUTPUT_DIR / "nearest_neighbors_recommender.joblib")
    (MODEL_OUTPUT_DIR / "recommender_features.json").write_text(
        json.dumps(recommender_feature_order, indent=2),
        encoding="utf-8",
    )
    recommender_catalog.to_csv(
        MODEL_OUTPUT_DIR / "recommender_catalog.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Rebuilt recommender artifacts saved to: {MODEL_OUTPUT_DIR}")
else:
    print("REBUILD_MODELS=False: accepted artifacts were loaded and no model file was written.")


## 17. Final findings

- The Extended Linear Regression is the accepted best popularity model with
  MAE 7.9820, RMSE 10.7308, and R² 0.7594.
- Adding year, duration, explicit status, key, and mode materially improves the
  popularity regression over audio features alone.
- The recommender uses audio similarity rather than popularity, so popularity
  remains display context instead of a neighbor feature.
- Validation confirms exact, deterministic Top N results without returning the
  seed track.
- SQL Server reporting is a separate group deliverable and is not claimed as
  complete by this notebook.


In [ ]:
final_summary = pd.DataFrame(
    {
        "result": [
            "Best regression model",
            "Accepted R²",
            "Recommender catalog rows",
            "Recommender features",
            "Smoke test",
        ],
        "value": [
            "Extended Linear Regression",
            "0.7594",
            f"{len(recommender_catalog):,}",
            str(len(recommender_feature_order)),
            "Model index 19611 returned exact Top 3",
        ],
    }
)
display(final_summary)
